# Fine-tune a small open model on our project data (LoRA) and export to GGUF
Setup: Notebook settings -> Accelerator: **GPU T4 x2** (or P100), **Internet: On**. Add your `train.jsonl` + `val.jsonl` as a Kaggle Dataset and attach it (Add Input). Then Run All.

In [ ]:
!pip install -q -U transformers peft datasets accelerate

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # use ONE GPU (avoids DataParallel problems with LoRA)
import glob, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
BASE = "Qwen/Qwen2.5-1.5B-Instruct"                # small, open, ungated. Any similar small instruct model works.
MAXLEN = 4096                                      # longest training example is ~2.2K tokens
train_p = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0]
val_p = glob.glob("/kaggle/input/**/val.jsonl", recursive=True)[0]
load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]
train, val = load(train_p), load(val_p)
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token

def enc(ex):                                       # train ONLY on the assistant's answer (prompt tokens masked)
    msgs = ex["messages"]
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    f = tok(prompt + msgs[-1]["content"] + tok.eos_token, add_special_tokens=False)["input_ids"][:MAXLEN]
    if len(p) >= len(f): return None
    return {"input_ids": f, "attention_mask": [1]*len(f), "labels": ([-100]*len(p) + f[len(p):])[:len(f)]}
data = [e for e in map(enc, train) if e]
print(len(data), "training examples")

def collate(b):
    m = max(len(x["input_ids"]) for x in b)
    pad = lambda k, v: [x[k] + [v]*(m - len(x[k])) for x in b]
    return {"input_ids": torch.tensor(pad("input_ids", tok.pad_token_id)),
            "attention_mask": torch.tensor(pad("attention_mask", 0)), "labels": torch.tensor(pad("labels", -100))}

In [ ]:
from peft import LoraConfig, get_peft_model
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map={"": 0})
model.gradient_checkpointing_enable(); model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
for p in model.parameters():
    if p.requires_grad: p.data = p.data.float()   # LoRA weights in fp32 (T4 has no bf16)
model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(output_dir="/kaggle/working/out", per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05, fp16=True,
        logging_steps=20, save_strategy="no", report_to="none", remove_unused_columns=False)
Trainer(model=model, args=args, train_dataset=data, data_collator=collate).train()

In [ ]:
model = model.merge_and_unload().half()
model.save_pretrained("/kaggle/working/merged"); tok.save_pretrained("/kaggle/working/merged")
model.eval()
for ex in val[:6]:                                  # quick sanity check on held-out examples
    msgs = ex["messages"]
    ids = tok(tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True), return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**ids, max_new_tokens=300, do_sample=False)
    print("QUESTION:", msgs[-2]["content"][-200:].replace("\n", " "))
    print("MODEL   :", tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))
    print("EXPECTED:", msgs[-1]["content"][:300].replace("\n", " "), "\n" + "-"*60)

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!pip install -q gguf sentencepiece protobuf
# if the next line complains about missing packages: pip install -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py /kaggle/working/merged --outfile /kaggle/working/training-output.gguf --outtype q8_0
!ls -lh /kaggle/working/*.gguf

This notebook exports a generic Q8 training artifact. The application runs the separately supplied v3 GGUF in-process using `OWN_MODEL_PATH`; it does not start an HTTP model server.